# Data Preprocessing & Feature Engineering

In [36]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/Steel_industry_data.csv')
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


## Chronological Ordering

In [49]:
# ordering date/time chronologically
df['date'] = pd.to_datetime(df['date'], format = '%d/%m/%Y %H:%M')
df = df.sort_values(by = 'date', ascending = True).reset_index(drop = True)
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


In [50]:
# checking the interval between date samples
df['date'].diff().unique()

<TimedeltaArray>
[NaT, '0 days 00:15:00']
Length: 2, dtype: timedelta64[us]

## Initial Data Checks

In [52]:
df['NSM_hrs'] = df.eval('NSM/3600')
df.groupby('Load_Type')['NSM_hrs'].mean()

Load_Type
Light_Load       7.884628
Maximum_Load    14.732261
Medium_Load     17.169554
Name: NSM_hrs, dtype: float64

In [53]:
# count of load type for each hour.
load_by_hour = pd.crosstab(df['NSM_hrs'].astype(int), df['Load_Type'])
load_by_hour

Load_Type,Light_Load,Maximum_Load,Medium_Load
NSM_hrs,,,
0,1460,0,0
1,1460,0,0
2,1460,0,0
3,1460,0,0
4,1460,0,0
5,1460,0,0
6,1460,0,0
7,1460,0,0
8,1460,0,0


## Time-Based Features

In [54]:
# creating time features
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute
df['month'] = df['date'].dt.month
df

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,NSM_hrs,hour,minute,month
0,2018-01-01 00:00:00,3.42,3.46,0.00,0.0,70.30,100.00,0,Weekday,Monday,Light_Load,0.00,0,0,1
1,2018-01-01 00:15:00,3.17,2.95,0.00,0.0,73.21,100.00,900,Weekday,Monday,Light_Load,0.25,0,15,1
2,2018-01-01 00:30:00,4.00,4.46,0.00,0.0,66.77,100.00,1800,Weekday,Monday,Light_Load,0.50,0,30,1
3,2018-01-01 00:45:00,3.24,3.28,0.00,0.0,70.28,100.00,2700,Weekday,Monday,Light_Load,0.75,0,45,1
4,2018-01-01 01:00:00,3.31,3.56,0.00,0.0,68.09,100.00,3600,Weekday,Monday,Light_Load,1.00,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,2018-12-31 22:45:00,3.82,4.54,0.00,0.0,64.38,100.00,81900,Weekday,Monday,Light_Load,22.75,22,45,12
35036,2018-12-31 23:00:00,3.85,4.86,0.00,0.0,62.10,100.00,82800,Weekday,Monday,Light_Load,23.00,23,0,12
35037,2018-12-31 23:15:00,3.74,3.74,0.00,0.0,70.71,100.00,83700,Weekday,Monday,Light_Load,23.25,23,15,12
35038,2018-12-31 23:30:00,3.78,3.17,0.07,0.0,76.62,99.98,84600,Weekday,Monday,Light_Load,23.50,23,30,12


## Lag Features

In [55]:
# creating lag features (previous observations)
df['lag_1'] = df['Usage_kWh'].shift(1) # 15 minutes ago
df['lag_2'] = df['Usage_kWh'].shift(2) # 30 minutes ago
df['lag_4'] = df['Usage_kWh'].shift(4) # 1 hour ago
df['lag_96'] = df['Usage_kWh'].shift(96) # 1 day ago
df['lag_672'] = df['Usage_kWh'].shift(672) # 1 week ago

In [56]:
df[df['date'] == '2018-11-07 00:00:00'] # marking out the row with unusual measurements

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,NSM_hrs,hour,minute,month,lag_1,lag_2,lag_4,lag_96,lag_672
29760,2018-11-07,0.0,0.0,0.0,0.0,0.0,0.0,0,Weekday,Wednesday,Light_Load,0.0,0,0,11,3.53,3.6,3.53,3.49,3.85


Summary: The data preprocessing stage involved correcting the chronological ordering of the dataset and confirming that the observations followed the expected 15-minute intervals. Time-based features including hour, minute and month were extracted, while lag features representing previous 15-minute, 30-minute, 1-hour, 1-day and 1-week energy usage were created to support forecasting. One observation with zero energy consumption and electrical measurements was identified as unusual; however, it was retained because there was insufficient evidence to confirm that it was a measurement error.

## Remove Rows Without Sufficient Lag History

In [58]:
df = df.drop(df.index[0:672]).reset_index(drop = True)
df

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,NSM_hrs,hour,minute,month,lag_1,lag_2,lag_4,lag_96,lag_672
0,2018-01-08 00:00:00,4.68,4.28,0.00,0.0,73.79,100.00,0,Weekday,Monday,Light_Load,0.00,0,0,1,3.42,3.64,3.64,3.28,3.42
1,2018-01-08 00:15:00,3.78,5.15,0.00,0.0,59.17,100.00,900,Weekday,Monday,Light_Load,0.25,0,15,1,4.68,3.42,3.24,3.13,3.17
2,2018-01-08 00:30:00,3.38,4.32,0.00,0.0,61.62,100.00,1800,Weekday,Monday,Light_Load,0.50,0,30,1,3.78,4.68,3.64,3.49,4.00
3,2018-01-08 00:45:00,3.31,4.25,0.00,0.0,61.45,100.00,2700,Weekday,Monday,Light_Load,0.75,0,45,1,3.38,3.78,3.42,3.64,3.24
4,2018-01-08 01:00:00,3.89,5.76,0.00,0.0,55.97,100.00,3600,Weekday,Monday,Light_Load,1.00,1,0,1,3.31,3.38,4.68,3.20,3.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34363,2018-12-31 22:45:00,3.82,4.54,0.00,0.0,64.38,100.00,81900,Weekday,Monday,Light_Load,22.75,22,45,12,3.67,3.24,3.42,3.10,3.89
34364,2018-12-31 23:00:00,3.85,4.86,0.00,0.0,62.10,100.00,82800,Weekday,Monday,Light_Load,23.00,23,0,12,3.82,3.67,3.42,3.02,3.85
34365,2018-12-31 23:15:00,3.74,3.74,0.00,0.0,70.71,100.00,83700,Weekday,Monday,Light_Load,23.25,23,15,12,3.85,3.82,3.24,2.95,3.85
34366,2018-12-31 23:30:00,3.78,3.17,0.07,0.0,76.62,99.98,84600,Weekday,Monday,Light_Load,23.50,23,30,12,3.74,3.85,3.67,2.99,3.82


In [62]:
model_df = df[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
    'lag_96','lag_672','Usage_kWh']]
model_df

,date,hour,minute,month,Day_of_week,WeekStatus,lag_1,lag_2,lag_4,lag_96,lag_672,Usage_kWh
0,2018-01-08 00:00:00,0,0,1,Monday,Weekday,3.42,3.64,3.64,3.28,3.42,4.68
1,2018-01-08 00:15:00,0,15,1,Monday,Weekday,4.68,3.42,3.24,3.13,3.17,3.78
2,2018-01-08 00:30:00,0,30,1,Monday,Weekday,3.78,4.68,3.64,3.49,4.00,3.38
3,2018-01-08 00:45:00,0,45,1,Monday,Weekday,3.38,3.78,3.42,3.64,3.24,3.31
4,2018-01-08 01:00:00,1,0,1,Monday,Weekday,3.31,3.38,4.68,3.20,3.31,3.89
...,...,...,...,...,...,...,...,...,...,...,...,...
34363,2018-12-31 22:45:00,22,45,12,Monday,Weekday,3.67,3.24,3.42,3.10,3.89,3.82
34364,2018-12-31 23:00:00,23,0,12,Monday,Weekday,3.82,3.67,3.42,3.02,3.85,3.85
34365,2018-12-31 23:15:00,23,15,12,Monday,Weekday,3.85,3.82,3.24,2.95,3.85,3.74
34366,2018-12-31 23:30:00,23,30,12,Monday,Weekday,3.74,3.85,3.67,2.99,3.82,3.78


## Chronological Train-Validation-Test Split

In [63]:
train_size = int(len(model_df)*0.7)
val_size = int(len(model_df)*0.15)

train = model_df.iloc[:train_size]
val = model_df.iloc[train_size:train_size+val_size]
test = model_df.iloc[train_size+val_size:]

In [65]:
print(train.shape)
print(val.shape)
print(test.shape)
print("Train:", train['date'].min(), "to", train['date'].max())
print("Validation:", val['date'].min(), "to", val['date'].max())
print("Test:", test['date'].min(), "to", test['date'].max())

(24057, 12)
(5155, 12)
(5156, 12)
Train: 2018-01-08 00:00:00 to 2018-09-15 14:00:00
Validation: 2018-09-15 14:15:00 to 2018-11-08 06:45:00
Test: 2018-11-08 07:00:00 to 2018-12-31 23:45:00


Rows without sufficient historical information were removed, leaving 34,368 observations. The dataset was then divided chronologically into training, validation and test sets to prevent future information from entering earlier periods, with 24,057, 5,155 and 5,156 observations respectively. The resulting datasets cover January–September, September–November and November–December 2018 respectively, providing a basis for forecasting.

## Save Processed Datasets

In [ ]:
train.to_csv('../data/train.csv', index=False)
val.to_csv('../data/validation.csv', index=False)
test.to_csv('../data/test.csv', index=False)